# 1. Introduction
![image.png](https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/refs/heads/sz-dev-2026/02-supervised-learning/ensembles/img/1-1.webp)


### 1. Bagging (Bootstrap Aggregating)
Bagging is an ensemble learning technique designed to improve the stability and accuracy of machine learning algorithms by reducing variance and helping to prevent overfitting. It works by generating multiple subsets of the original training data through random sampling with replacement (bootstrapping). A base model is then trained independently on each subset, and their final predictions are aggregated—typically by majority voting for classification tasks or averaging for regression tasks.

* **Generic bagging**
  * We will use logistic regression as the base learner, and compare the predictive performance of:
    * Manual coding from scratch
    * Using `BaggingClassifier` with logistic regression
* **Use decision tree as base learner (random forest)**
  * We will compare the predictive performance of three models:
    * Manual coding from scratch
    * Sklearn Random Forest with sub-featuring (feature sampling)
    * Sklearn Random Forest without sub-featuring 

### 2. Boosting
Boosting is a sequential ensemble learning technique that combines multiple weak learners to create a single strong learner. Unlike bagging, which builds models independently in parallel, boosting builds models one after another. Each new model specifically focuses on correcting the errors made by the previous models—often by assigning higher weights to misclassified data points. The final prediction is a weighted combination of all the individual model predictions, effectively reducing both bias and variance.

* We will use a decision tree as the base learner, specifically focusing on **Gradient Boosting**. 
* We will compare the predictive performance of two models:
  * Manual implementation from scratch
  * Sklearn version (`GradientBoostingClassifier`)

### 3. Voting
Voting is a straightforward but powerful ensemble method that combines the predictions of multiple distinct, independent base models (which can be of entirely different types, like combining a Support Vector Machine, a Decision Tree, and Logistic Regression). For classification, it outputs a final prediction based on either "hard voting" (the class that receives the majority of votes) or "soft voting" (the class with the highest average predicted probability). This approach leverages the diverse strengths of different algorithms to achieve better overall accuracy.

* **Implementation Comparison:**
  * Manual implementation from scratch
  * Sklearn version (`VotingClassifier` )

## 1.1 Loading dataset

We will again be using the digit dataset

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

# 1. Load the dataset
digits = load_digits()

# 2. Extract features (X) and target labels (y)
# X contains the flattened 8x8 images (64 features per sample)
# y contains the actual true digit (0 through 9)
X = digits.data
y = digits.target

# 3. Split into training and testing sets 
# Using a fixed random_state ensures your comparisons are fair and reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

# Optional: Print the shapes to verify
print(f"Total dataset shape: {X.shape}")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

plt.figure(1, figsize=(3, 3))
plt.imshow(digits.images[9], cmap=plt.cm.gray_r, interpolation="nearest")
plt.show()

# 2. Bagging

* Definition and Purpose of Bagging:

  - Bagging, short for Bootstrap Aggregating, is an ensemble learning technique that aims to improve the accuracy and robustness of predictive models.
  - The primary purpose of bagging is to reduce variance and combat overfitting, which are common challenges in machine learning.
  - By combining predictions from multiple base learners, bagging reduces the risk of relying too heavily on any one model's idiosyncrasies.

* How Bagging Works:

  * Bootstrapping and Creating Subsets:
    - Bagging starts by creating multiple subsets of the original training data through bootstrapping.
    - Bootstrapping involves randomly sampling data with replacement, resulting in diverse subsets with the same size as the original dataset.


  * Training Multiple Base Learners:
    - Each subset is used to train an independent base learner (model), such as decision trees, SVMs, or neural networks.
    - These base learners are trained independently, and their predictions are combined later to form the ensemble.


  * Aggregating Predictions:
    - Once all base learners are trained, the final prediction is made by aggregating their individual predictions.
    - For regression tasks, predictions are averaged, while for classification tasks, majority voting is used to decide the final prediction.

![image.png](https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/refs/heads/sz-dev-2026/02-supervised-learning/ensembles/img/2.png)

* Advantages of Bagging:

  * Reducing Variance and Overfitting:
    - Bagging's main advantage is its ability to reduce variance and combat overfitting.
    - By combining predictions from diverse models, it results in a more balanced and stable ensemble model.


  * Robustness to Noisy Data and Outliers:
    - Bagging is robust to noisy data and outliers, as it diminishes the impact of individual data points through aggregation.
    - Outliers are less likely to affect the overall prediction due to the averaging or voting process.


* Common Algorithms Using Bagging:

  * Random Forest:
    - Random Forest is one of the most popular bagging algorithms that uses decision trees as base learners.
    - It builds a large number of decision trees, and each tree is trained on a bootstrapped subset of the data.
    - The final prediction is made by aggregating the predictions of all individual trees.

  * Bagged Decision Trees:
     - Bagged Decision Trees, or simply Bagging with decision trees, is a straightforward bagging approach.
     - It applies the same concept as Random Forest but with a smaller number of trees (often not enough to be considered a forest).

  * Bagged SVMs (Support Vector Machines):
    - Bagging can also be applied to other base learners like Support Vector Machines (SVMs).
    - Bagged SVMs utilize subsets of data for training multiple SVM models, and their predictions are aggregated to make the final prediction.


### 2.1 Example 1: Use Logistic Regression as base learner
Bagging linear models like Logistic Regression often yields less dramatic improvements than bagging trees because linear models are generally "low variance" (they don't overfit as easily as deep trees). However, it is a perfect exercise for understanding the mechanics.

First we look at the manual implemetnation:

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from scipy.stats import mode

class LogitBagging:
    def __init__(self, n_estimators=100):
        self.n_estimators = n_estimators
        self.models = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        for _ in range(self.n_estimators):
            # Bootstrap sampling
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot, y_boot = X[indices], y[indices]
            
            # Train Logistic Regression
            model = LogisticRegression(max_iter=1000)
            model.fit(X_boot, y_boot)
            self.models.append(model)

    def predict(self, X):
        preds = np.array([model.predict(X) for model in self.models])
        majority_votes, _ = mode(preds, axis=0)
        return majority_votes.flatten()

m_logit_bag = LogitBagging(n_estimators=100)
m_logit_bag.fit(X_train, y_train)


**Use `sklearn`'s implementation**

In [ ]:
from sklearn.ensemble import BaggingClassifier

sk_logit_bag = BaggingClassifier(
    estimator=LogisticRegression(max_iter=1000), 
    n_estimators=100, 
)
sk_logit_bag.fit(X_train, y_train)


**Comparing accuracy**

Now let's compare the model accuracy between the two models using visualization. 

In [ ]:
from sklearn.metrics import accuracy_score



from sklearn.metrics import accuracy_score

m_logit_acc = accuracy_score(y_test, m_logit_bag.predict(X_test))
sk_logit_acc = accuracy_score(y_test, sk_logit_bag.predict(X_test))





categories = ['Manual implementation', 'sklearn\'s implementation',]
values = [m_logit_acc, sk_logit_acc]

plt.figure(figsize=(7, 5))
bars = plt.bar(categories, values, color=['skyblue', 'lightgreen', 'pink'])

# Add values on top of the bars
plt.bar_label(bars, padding=3)

plt.title('Comparison accuracy of the bagging classifier')
plt.ylabel('Values')

plt.ylim(0, max(values) * 1.2) # Add some space above the bars for the labels



### 2.2 Example 2: Use Decision Tree as base learner


**Manual Coding:**

To truly manually code a Random Forest, you have to inject randomness into the feature selection during the tree-building process. Since writing a tree algorithm from scratch is incredibly complex, we can "cheat" by using Sklearn's base tree and forcing it to subset features, wrapping it in our manual bootstrap loop.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

class ManualRandomForest:
    def __init__(self, n_estimators=100):
        self.n_estimators = n_estimators
        self.trees = []

    def fit(self, X, y):
        n_samples = X.shape[0]
        for _ in range(self.n_estimators):
            indices = np.random.choice(n_samples, size=n_samples, replace=True)
            # max_features='sqrt' introduces the RF feature-subsetting behavior
            tree = DecisionTreeClassifier(max_features='sqrt') 
            tree.fit(X[indices], y[indices])
            self.trees.append(tree)

    def predict(self, X):
        preds = np.array([tree.predict(X) for tree in self.trees])
        majority_votes, _ = mode(preds, axis=0)
        return majority_votes.flatten()
    
rf_man = ManualRandomForest()
rf_man.fit(X_train, y_train)


**Use `sklearn`'s implementation (with feature subsetting)**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_sk = RandomForestClassifier(n_estimators=100, max_features='sqrt')
rf_sk.fit(X_train, y_train)

**Use `sklearn`'s implementation (full feature)**

In [ ]:
sk_rf_nosub = RandomForestClassifier(n_estimators=100, max_features=None)
sk_rf_nosub.fit(X_train, y_train)


**Comparing accuracy**

Now let's compare the model accuracy among the three models using visualization. 

In [ ]:


from sklearn.metrics import accuracy_score

rf_man_acc = accuracy_score(y_test, rf_man.predict(X_test))
rf_sk_acc = accuracy_score(y_test, rf_sk.predict(X_test))
sk_rf_nosub = accuracy_score(y_test, sk_rf_nosub.predict(X_test))





categories = ['Manual implementation', 'sklearn\'s implementation\n(with subset features)', 'sklearn\'s implementation\n(full features set)']
values = [rf_man_acc, rf_sk_acc, sk_rf_nosub]

plt.figure(figsize=(10, 6))
bars = plt.bar(categories, values, color=['skyblue', 'lightgreen', 'pink'])

# Add values on top of the bars
plt.bar_label(bars, padding=3)

plt.title('Comparison accuracy of the bagging classifier')
plt.ylabel('Values')

plt.ylim(0, max(values) * 1.2) # Add some space above the bars for the labels



# 3. Boosting

* Definition and Purpose of Boosting:

  - Boosting is an ensemble learning technique that focuses on creating a strong learner by combining multiple weak learners iteratively.
  - The main purpose of boosting is to improve the accuracy and performance of predictive models by giving more emphasis to misclassified instances during training.
  - Boosting learns from its mistakes in an adaptive manner, continually refining its predictions to achieve high accuracy on complex datasets.


* How Boosting Works:

  1. Sequential Learning and Weighted Misclassifications:
       - Boosting works in a sequential manner, where each base learner is trained based on the performance of the previous ones.
       - During training, it assigns higher weights to misclassified instances, making them more influential in subsequent iterations.


  2. Iterative Training of Base Learners:
       - In each iteration, a new base learner is trained to correct the mistakes made by the ensemble so far.
       - The base learners are typically weak models (e.g., shallow decision trees) to avoid overfitting and maintain interpretability.


  3. Emphasizing Difficult Instances:
       - Boosting focuses on challenging instances that are frequently misclassified by previous base learners.
       - By repeatedly emphasizing these difficult instances, boosting ensures that the ensemble model pays more attention to them and gradually improves its performance.

![image.png](https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/refs/heads/sz-dev-2026/02-supervised-learning/ensembles/img/3.png)


*  Advantages of Boosting:

   1. Adaptive Learning and High Accuracy:
      - Boosting's adaptive learning approach allows it to learn from misclassifications and significantly improve predictive accuracy.
      - It is particularly effective in handling complex relationships in data, making it suitable for various real-world applications.


   2. Model Versatility and Feature Importance:
      - Boosting can be applied with various base learners, such as decision trees, SVMs, and neural networks.
      - Additionally, many boosting algorithms provide feature importance scores, enabling us to identify the most influential features in the model's decision-making process.


* Common Algorithms Using Boosting:

  1. AdaBoost (Adaptive Boosting):
     - AdaBoost is one of the earliest and most popular boosting algorithms.
     - It assigns higher weights to misclassified instances and combines the predictions of weak learners to create a strong ensemble model.
     - AdaBoost is suitable for both classification and regression tasks.


  2. Gradient Boosting Machines (GBM):
     - GBM builds base learners sequentially, focusing on the gradients of the loss function to optimize the model's performance.
     - It uses a process called gradient descent to minimize the errors in each iteration.
     - GBM is widely used for regression and classification tasks and is known for its high accuracy and flexibility.


  3. XGBoost and LightGBM:
     - XGBoost and LightGBM are optimized implementations of gradient boosting that are efficient and scalable.
     - They utilize advanced techniques like parallel processing and tree-pruning to achieve better performance.
     - These algorithms are popular in data science competitions and real-world applications due to their speed and accuracy.


## 3.1 Example implementation

Boosting trains models sequentially. If we assume a regression task (or a classification task mapped to probabilities), each new tree fits to the residual errors ($r_i = y_i - \hat{y}_i$) of the combined previous trees.

**Manual Implementation (Regression/Residual logic):**

In [ ]:
from sklearn.tree import DecisionTreeRegressor


import numpy as np

class ManualGradientBoostingClassifier:
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.initial_pred = None

    def fit(self, X, y):
        # 1. Convert y into a One-Hot Encoded Matrix (Shape: N x 10)
        n_classes = 10
        Y_oh = np.zeros((len(y), n_classes))
        Y_oh[np.arange(len(y)), y] = 1
        
        # 2. Initial prediction is the baseline probability of each class
        self.initial_pred = np.mean(Y_oh, axis=0) # Shape: (10,)
        current_preds = np.tile(self.initial_pred, (X.shape[0], 1))
        
        for _ in range(self.n_estimators):
            # 3. Calculate residuals for ALL 10 classes simultaneously
            residuals = Y_oh - current_preds
            
            # 4. Multi-output regression tree 
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            
            current_preds += self.learning_rate * tree.predict(X)

    def predict(self, X):
        # 5. Tile initial predictions to shape (N_test, 10)
        preds = np.tile(self.initial_pred, (X.shape[0], 1))
        
        for tree in self.trees:
            # Shape (N_test, 10) += Shape (N_test, 10) -> Perfect Match!
            preds += self.learning_rate * tree.predict(X)
            
        # 6. Convert continuous probabilities back to an integer class (0-9)
        return np.argmax(preds, axis=1)
    
m_gb = ManualGradientBoostingClassifier(n_estimators=100, learning_rate=0.1,)
m_gb.fit(X_train, y_train)

**Use `sklearn`'s implementation**

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

sk_gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
sk_gb.fit(X_train, y_train)


**Comparing accuracy**

Now let's compare the model accuracy between the two models using visualization. 

In [ ]:

from sklearn.metrics import accuracy_score

man_vote_acc = accuracy_score(y_test, m_gb.predict(X_test))
sk_vote_acc = accuracy_score(y_test, sk_gb.predict(X_test))





categories = ['Manual implementation', 'sklearn\'s implementation']
values = [man_vote_acc, sk_vote_acc]

plt.figure(figsize=(6, 4))
bars = plt.bar(categories, values, color=['skyblue', 'lightgreen'])

# Add values on top of the bars
plt.bar_label(bars, padding=3)

plt.title('Comparison accuracy of the boosting  classifier')
plt.ylabel('Values')

plt.ylim(0, max(values) * 1.2) # Add some space above the bars for the labels




# 4. Voting

* Definition and Purpose of Voting:

  - Voting is an ensemble learning technique that combines the predictions of multiple distinct machine learning models to make a final, unified prediction.
  - The primary purpose of voting is to leverage the unique strengths of different algorithms, effectively balancing out their individual weaknesses to improve overall predictive accuracy.
  - Unlike bagging, which trains the *same* type of model on different subsets of data, voting typically trains *different* types of models (heterogeneous base learners) on the exact same original dataset.

* How Voting Works:

  * Training Diverse Base Learners:
    - Voting starts by selecting multiple different algorithms (for example, a Logistic Regression model, a Random Forest, and a Support Vector Machine).
    - Each of these distinct models is trained independently on the **complete** training dataset.

  * Generating Independent Predictions:
    - Once trained, a new, unseen data point is passed through every individual model in the ensemble.
    - Each base learner independently processes the data and generates its own prediction or set of class probabilities.

  * Aggregating Predictions:
    - The final output is determined by combining these individual results.
    - For classification tasks, this is done by tallying the predicted labels or averaging the predicted probabilities. For regression tasks, the continuous numerical predictions are simply averaged together.

![image.png](https://raw.githubusercontent.com/uoft-ml-bootcamp/tutorials/refs/heads/sz-dev-2026/02-supervised-learning/ensembles/img/4.png)

* Advantages of Voting:

  * Balancing Individual Weaknesses:
    - Every machine learning algorithm makes different mathematical assumptions about the data. Voting ensures that an error caused by one model's specific "blind spot" can be outvoted and corrected by the accurate predictions of the others.
    - It effectively creates a "wisdom of the crowd" scenario, resulting in a more generalized and highly stable predictive model.

  * Flexibility and Customization:
    - Voting is highly flexible, allowing practitioners to hand-pick and combine only the absolute best-performing models discovered during their initial experimentation phase.
    - It provides a straightforward way to boost performance without requiring complex mathematical tuning for the ensemble mechanism itself.

* Common Variations of Voting:

  * Hard Voting (Majority Rule):
    - In hard voting, every base learner outputs a strict, discrete class label (e.g., "Digit 3" or "Digit 8").
    - The ensemble operates like a simple democracy: the class that receives the majority of the votes across all models becomes the final overall prediction.

  * Soft Voting (Probability Averaging):
    - In soft voting, base learners must be capable of outputting a confidence score or probability for each class (e.g., "90% chance of Digit 3").
    - The ensemble averages these probabilities across all models and selects the class with the highest average score. It often yields better results than hard voting because it factors in the *certainty* of each model.

  * Weighted Voting:
    - Weighted voting is an enhancement (applicable to both hard and soft voting) where certain models are granted more voting power than others.
    - If one specific model is known to be historically more accurate or trustworthy on a specific dataset, its prediction is multiplied by a higher weight before the final aggregation occurs.



## Example implementation

**Manual implementation**

In this manual implementation, we will use four base learners: `LogisticRegression`, `RandomForestClassifier`, `GradientBoostingClassifier`, `KNeighborsClassifier`

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier


# Step 1: we first need to define the base learner
model_lr = LogisticRegression(max_iter=3000, random_state=42)
model_rf = RandomForestClassifier(n_estimators=100, random_state=42) 
model_gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
model_knn = KNeighborsClassifier(n_neighbors=5)

# ...and then we put them into a list as the input to the voting classifier
estimators_list = [
    ('lr', model_lr), 
    ('rf', model_rf), 
    ('gb', model_gb),
    ('knn', model_knn)
]

# Step 2: we cna now build the classifier
class ManualVotingClassifier:
    def __init__(self, estimators, voting='soft'):
        self.estimators = estimators
        self.voting = voting
        self.classes_ = None

    def fit(self, X, y):
        # Fit each base model
        for name, model in self.estimators:
            model.fit(X, y)
            
        # Store the unique classes from the first model (digits 0-9)
        self.classes_ = self.estimators[0][1].classes_
        return self

    def predict(self, X):
        if self.voting == 'hard':
            # Collect predictions: shape = (num_models, num_samples)
            preds = np.asarray([model.predict(X) for _, model in self.estimators])
            
            # Find the mode (most common prediction) for each column (sample)
            majority_votes = np.apply_along_axis(
                lambda x: np.bincount(x).argmax(), 
                axis=0, 
                arr=preds
            )
            return majority_votes
            
        elif self.voting == 'soft':
            # Average the probabilities and return the class with the highest average
            avg_probs = self.predict_proba(X)
            return self.classes_[np.argmax(avg_probs, axis=1)]

    def predict_proba(self, X):
        if self.voting == 'hard':
            raise AttributeError("predict_proba is not available when voting='hard'")
        
        # Collect probabilities: shape = (num_models, num_samples, 10 classes)
        probs = np.asarray([model.predict_proba(X) for _, model in self.estimators])
        return np.mean(probs, axis=0)

# Step 3: initiate our own classifier and train it!

manual_voting_clf = ManualVotingClassifier(estimators=estimators_list, voting='soft')
manual_voting_clf.fit(X_train, y_train)

**Use `sklearn`'s implementation**

In [ ]:
sklearn_voting_clf = VotingClassifier(estimators=estimators_list, voting='soft')
sklearn_voting_clf.fit(X_train, y_train)


**Comparing accuracy**

Now let's compare the model accuracy between the two models using visualization. 

In [ ]:
from sklearn.metrics import accuracy_score

man_vote_acc = accuracy_score(y_test, manual_voting_clf.predict(X_test))
sk_vote_acc = accuracy_score(y_test, sklearn_voting_clf.predict(X_test))





categories = ['Manual implementation', 'sklearn\'s implementation']
values = [man_vote_acc, sk_vote_acc]

plt.figure(figsize=(6, 4))
bars = plt.bar(categories, values, color=['skyblue', 'lightgreen'])

# Add values on top of the bars
plt.bar_label(bars, padding=3)

plt.title('Comparison accuracy of the voting classifier')
plt.ylabel('Values')

plt.ylim(0, max(values) * 1.2) # Add some space above the bars for the labels

